# install dependencies

In [32]:
import pandas as pd
import numpy as np
import re
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LassoCV
from sklearn.model_selection import KFold
import statsmodels.api as sm

# Phase1 Data Preparation

In [33]:
def calculate_cir_series(
    outcome,
    file_name,
    data_path='../../result/occ/analysis',
    horizon_start=1,
    horizon_end=36,
    series_name=None
):
    file_path = f"{data_path.rstrip('/\\')}/{file_name}"
    df = pd.read_csv(file_path)

    if series_name is None:
        series_name = f"CIR_{horizon_end}"

    df_outcome = df[df['outcome'] == outcome].copy()
    df_outcome['horizon'] = pd.to_numeric(df_outcome['horizon'])
    df_outcome = df_outcome.sort_values('horizon').reset_index(drop=True)

    group_cols = [col for col in df_outcome.columns if col.startswith('group')]

    cir_dict = {}
    for col in group_cols:
        irf_window = df_outcome[df_outcome['horizon'].between(horizon_start, horizon_end)][col]
        cir_dict[col] = irf_window.sum()

    cir_series = pd.Series(cir_dict, name=series_name)
    return cir_series

In [34]:
def build_y_series_from_mapping(
    cir_series,
    file_name,
    data_path='../../result/mapping',
    sheet_name='Sheet1',
    usecols='A,E',
    series_name=None
):
    mapping_path = f"{data_path.rstrip('/\\')}/{file_name}"
    df_map = pd.read_excel(mapping_path, sheet_name=sheet_name, usecols=usecols, header=0)
    df_map.columns = ['occ1990', 'SOC-2018']

    if series_name is None:
        series_name = cir_series.name if cir_series.name is not None else 'value'

    df_map['occ1990'] = pd.to_numeric(df_map['occ1990'], errors='coerce').astype('Int64')
    df_map['SOC-2018'] = df_map['SOC-2018'].astype(str).str.strip()
    df_map = df_map.dropna(subset=['occ1990', 'SOC-2018'])

    def occ1990_to_group(occ):
        if 3 <= occ <= 37: return 1
        elif 43 <= occ <= 200: return 2
        elif 203 <= occ <= 235: return 3
        elif 243 <= occ <= 283: return 4
        elif 303 <= occ <= 389: return 5
        elif 405 <= occ <= 469: return 6
        elif (473 <= occ <= 498) or (558 <= occ <= 599) or (614 <= occ <= 617): return 7
        elif (503 <= occ <= 549) or (628 <= occ <= 699): return 8
        elif (703 <= occ <= 799) or (803 <= occ <= 889): return 9
        return np.nan

    df_map['group'] = df_map['occ1990'].apply(occ1990_to_group)

    group_to_value = {}
    for idx, val in cir_series.items():
        match = re.search(r'group(\d+)', str(idx))
        if match:
            group_to_value[int(match.group(1))] = val

    df_map[series_name] = df_map['group'].map(group_to_value)
    df_final = df_map.drop_duplicates(subset='SOC-2018', keep='first')
    y_series = df_final.set_index('SOC-2018')[series_name].dropna()

    return y_series

In [35]:
def load_and_prepare_onet_data(
    y_series,
    file_name,
    data_path='../../data/ONET',
    usecols=[0, 1, 4, 5, 7],
    scale_id='LV'
):
    file_path = f"{data_path.rstrip('/\\')}/{file_name}"
    df = pd.read_excel(file_path, usecols=usecols, header=0)
    df.columns = ['SOC_Code', 'Sub_Code', 'Element_Name', 'Scale_ID', 'Data_Value']

    df['SOC_Code'] = df['SOC_Code'].astype(str).str.strip()
    df['Element_Name'] = df['Element_Name'].astype(str).str.strip()
    df['Scale_ID'] = df['Scale_ID'].astype(str).str.strip().str.upper()
    df['Sub_Code'] = df['Sub_Code'].astype(str).str.strip().str.zfill(2)

    soc_elem_means = df.groupby(['SOC_Code', 'Element_Name'])['Data_Value'].mean().reset_index()
    soc_elem_means.rename(columns={'Data_Value': 'Mean_Val'}, inplace=True)

    df = df.merge(soc_elem_means, on=['SOC_Code', 'Element_Name'], how='left')
    df.loc[df['Sub_Code'] == '00', 'Data_Value'] = df.loc[df['Sub_Code'] == '00', 'Mean_Val']

    df = df[df['Sub_Code'] == '00'].copy()
    df.drop(columns=['Mean_Val', 'Sub_Code'], inplace=True)

    df = df[df['Scale_ID'] == scale_id].copy()
    df = df.dropna(subset=['Data_Value'])

    df_wide = df.pivot_table(
        index='SOC_Code',
        columns='Element_Name',
        values='Data_Value',
        aggfunc='mean'
    )
    df_wide = df_wide.astype(float)
    df_wide = df_wide.fillna(df_wide.median())

    if df_wide.shape[1] == 0:
        raise ValueError("df_wide has no columns; check Sub_Code and Scale_ID filters above.")

    scaler = StandardScaler()
    X_df = pd.DataFrame(
        scaler.fit_transform(df_wide),
        columns=df_wide.columns,
        index=df_wide.index
    )

    aligned_idx = X_df.index.intersection(y_series.index)
    X = X_df.loc[aligned_idx].values
    y_aligned = y_series.loc[aligned_idx].values

    return X_df, aligned_idx, X, y_aligned

# Phase2: LASSO Estimation

In [36]:
def run_lasso_selection(
    X,
    y_aligned,
    feature_names,
    top_n=10,
    n_splits=10,
    random_state=42,
    max_iter=5000
):
    feature_names = pd.Index(feature_names)
    cv_strategy = KFold(n_splits=n_splits, shuffle=True, random_state=random_state)

    lasso_cv = LassoCV(
        alphas=None,
        cv=cv_strategy,
        max_iter=max_iter,
        random_state=random_state,
        n_jobs=-1
    )
    lasso_cv.fit(X, y_aligned)

    best_alpha = lasso_cv.alpha_
    best_coefs = lasso_cv.coef_
    cv_mse_path = lasso_cv.mse_path_
    cv_mean_mse = cv_mse_path.mean(axis=1)

    nonzero_mask = best_coefs != 0
    nonzero_idx = np.where(nonzero_mask)[0]
    n_nonzero = len(nonzero_idx)

    nonzero_idx_sorted = nonzero_idx[np.argsort(np.abs(best_coefs[nonzero_idx]))[::-1]]
    top_idx = nonzero_idx_sorted[:top_n]

    top_names = feature_names.take(top_idx).tolist()
    top_coefs = best_coefs[top_idx]

    selected_mask = np.zeros(len(feature_names), dtype=bool)
    selected_mask[top_idx] = True

    if n_nonzero < top_n:
        print(f"LASSO only selects {n_nonzero} non-zero variables, fewer than top_n={top_n}, actually using {n_nonzero} variables")

    return {
        'lasso_cv': lasso_cv,
        'best_alpha': best_alpha,
        'best_coefs': best_coefs,
        'cv_mse_path': cv_mse_path,
        'cv_mean_mse': cv_mean_mse,
        'feature_names': feature_names,
        'top_idx': top_idx,
        'top_names': top_names,
        'top_coefs': top_coefs,
        'selected_mask': selected_mask
    }

In [37]:
def lasso_stability_check(
    X,
    y_aligned,
    feature_names,
    top_n=10,
    n_boots=100,
    n_splits=10,
    freq_threshold=0.9,
    random_state=42,
    max_iter=5000
):
    feature_names = pd.Index(feature_names)
    rng = np.random.default_rng(random_state)
    n = len(y_aligned)
    selection_counts = np.zeros(len(feature_names))

    for i in range(n_boots):
        idx = rng.integers(0, n, size=n)
        X_b, y_b = X[idx], y_aligned[idx]
        cv = KFold(n_splits=n_splits, shuffle=True, random_state=int(rng.integers(9999)))
        m = LassoCV(cv=cv, max_iter=max_iter, n_jobs=-1).fit(X_b, y_b)
        selection_counts += (m.coef_ != 0).astype(int)

    freq = pd.Series(selection_counts / n_boots, index=feature_names)
    freq = freq.sort_values(ascending=False)

    # 稳定变量：频率 >= freq_threshold
    stable_features = freq[freq >= freq_threshold].index.tolist()
    # 在稳定变量里再截断到 top_n
    final_features = stable_features[:top_n]

    print(f"=== Bootstrap 稳定性检验 (n_boots={n_boots}, threshold={freq_threshold}) ===")
    print(f"频率 >= {freq_threshold} 的变量: {len(stable_features)} 个")
    print(f"频率 >= 0.8 的变量 (高稳定): {(freq >= 0.8).sum()} 个")
    print(f"最终进入 OLS 的变量: {len(final_features)} 个\n")
    print("选中频率 top 15:")
    print(freq.head(15).round(3).to_string())

    # 构建 selected_mask（基于稳定变量，而非单次 LASSO）
    selected_mask = np.zeros(len(feature_names), dtype=bool)
    for name in final_features:
        selected_mask[feature_names.get_loc(name)] = True

    return {
        'freq': freq,
        'stable_features': stable_features,
        'final_features': final_features,
        'selected_mask': selected_mask
    }

In [38]:
def calculate_post_lasso_r2(X, y_aligned, selected_mask, selected_feature_names):
    X_selected = X[:, selected_mask]
    X_selected_const = sm.add_constant(X_selected)

    ols_model = sm.OLS(y_aligned, X_selected_const).fit(cov_type='HC3')
    r_squared = ols_model.rsquared
    r_squared_adj = ols_model.rsquared_adj
    ci = ols_model.conf_int()

    ols_results_df = pd.DataFrame({
        'O*NET_Activity': selected_feature_names,
        'OLS_Coefficient': ols_model.params[1:],
        'CI_Lower': ci[1:, 0],
        'CI_Upper': ci[1:, 1],
        'Std_Error': ols_model.bse[1:],
        'P_Value': ols_model.pvalues[1:],
        'Sig_10%': ols_model.pvalues[1:] < 0.10,
        'Sig_5%': ols_model.pvalues[1:] < 0.05
    }).sort_values('OLS_Coefficient', key=abs, ascending=False)

    return {
        'ols_model': ols_model,
        'r_squared': r_squared,
        'r_squared_adj': r_squared_adj,
        'ols_results_df': ols_results_df
    }

# Main

In [39]:
# choose horizon here
horizon_end = 12

# Step 1. build CIR series
cir_series = calculate_cir_series(
    outcome='hourly_rate',
    file_name='merged_occ_irf_trajectories.csv',
    horizon_start=1,
    horizon_end=horizon_end
)

print(cir_series)

# Step 2. build y_series from mapping
y_series = build_y_series_from_mapping(
    cir_series=cir_series,
    file_name='mapping_done.xlsx'
)

print(y_series.head())

# Step 3. prepare O*NET data
X_df, aligned_idx, X, y_aligned = load_and_prepare_onet_data(
    y_series=y_series,
    file_name='Work Activities.xlsx'
)

print(f"X shape: {X.shape}")
print(f"Aligned samples: {len(aligned_idx)}")

# Step 4. run LASSO
lasso_results = run_lasso_selection(
    X=X,
    y_aligned=y_aligned,
    feature_names=X_df.loc[aligned_idx].columns
)

print(f"Best alpha: {lasso_results['best_alpha']:.4f}")
print(f"Number of non-zero coefficients: {np.sum(lasso_results['best_coefs'] != 0)}")
print("Top 10 O*NET Activities:")
for name, coef in zip(lasso_results['top_names'], lasso_results['top_coefs']):
    sign = "+" if coef > 0 else "-"
    print(f"  {sign} {name:<45} | coefficient: {coef:.4f}")

stability_results = lasso_stability_check(
    X=X,
    y_aligned=y_aligned,
    feature_names=X_df.loc[aligned_idx].columns
)

# Step 5. post-lasso OLS and R2
post_lasso_results = calculate_post_lasso_r2(
    X=X,
    y_aligned=y_aligned,
    selected_mask=stability_results['selected_mask'],
    selected_feature_names=stability_results['final_features']
)

print("\nPost-Lasso OLS results:")
print(f"R^2: {post_lasso_results['r_squared']:.4f} | Adjusted R^2: {post_lasso_results['r_squared_adj']:.4f}")
print(f"Sample Size (n): {post_lasso_results['ols_model'].nobs}")

print("\nPost-Lasso OLS coefficients:")
print(post_lasso_results['ols_results_df'].to_string(
    index=False,
    formatters={
        'OLS_Coefficient': '{:+.4f}'.format,
        'Std_Error': '({:.4f})'.format,
        'P_Value': '{:.4f}'.format
    }
))


group1_Managerial                      -0.026237
group2_Professional_specialty          -0.012740
group3_High_tech                       -0.065769
group4_Sales                           -0.061173
group5_Administrative_support          -0.033067
group6_Service                         -0.010198
group7_Farming_forestry_construction   -0.016141
group8_Precision_production_repair     -0.109801
group9_Machine_operators_transport     -0.002255
Name: CIR_12, dtype: float64
SOC-2018
11-1011   -0.026237
11-3012   -0.026237
33-1012   -0.026237
11-3031   -0.026237
11-3111   -0.026237
Name: CIR_12, dtype: float64
X shape: (609, 41)
Aligned samples: 609
Best alpha: 0.0004
Number of non-zero coefficients: 29
Top 10 O*NET Activities:
  - Repairing and Maintaining Mechanical Equipment | coefficient: -0.0132
  - Organizing, Planning, and Prioritizing Work   | coefficient: -0.0103
  + Developing Objectives and Strategies          | coefficient: 0.0062
  - Selling or Influencing Others                 | c

In [44]:
all_horizon_results = {}

for horizon_end in [6, 12, 36]:
    print(f"\n{'='*60}")
    print(f"Horizon {horizon_end}")
    print(f"{'='*60}")

    cir_series = calculate_cir_series(
        outcome='unemployment',
        file_name='merged_occ_irf_trajectories.csv',
        horizon_end=horizon_end
    )

    y_series = build_y_series_from_mapping(
        cir_series=cir_series,
        file_name='mapping_done.xlsx'
    )

    X_df, aligned_idx, X, y_aligned = load_and_prepare_onet_data(
        y_series=y_series,
        file_name='Abilities.xlsx'
    )

    # 单次 LASSO（诊断用）
    lasso_results = run_lasso_selection(
        X=X,
        y_aligned=y_aligned,
        feature_names=X_df.loc[aligned_idx].columns
    )
    print(f"Best alpha: {lasso_results['best_alpha']:.4f} | Non-zero: {np.sum(lasso_results['best_coefs'] != 0)}")
    # Bootstrap 稳定性检验
    stability_results = lasso_stability_check(
        X=X,
        y_aligned=y_aligned,
        feature_names=X_df.loc[aligned_idx].columns,
        freq_threshold=0.9
    )

    # Post-LASSO OLS
    post_lasso_results = calculate_post_lasso_r2(
        X=X,
        y_aligned=y_aligned,
        selected_mask=stability_results['selected_mask'],
        selected_feature_names=stability_results['final_features']
    )

    print(f"\nR^2: {post_lasso_results['r_squared']:.4f} | Adj R^2: {post_lasso_results['r_squared_adj']:.4f}")
    print(f"Sample size: {int(post_lasso_results['ols_model'].nobs)}")
    print("\nOLS coefficients:")
    print(post_lasso_results['ols_results_df'].to_string(
        index=False,
        formatters={
            'OLS_Coefficient': '{:+.4f}'.format,
            'Std_Error': '({:.4f})'.format,
            'P_Value': '{:.4f}'.format
        }
    ))

    # 结果存起来方便后面跨horizon比较
    all_horizon_results[horizon_end] = {
        'freq': stability_results['freq'],
        'final_features': stability_results['final_features'],
        'ols_df': post_lasso_results['ols_results_df'],
        'r2': post_lasso_results['r_squared'],
        'adj_r2': post_lasso_results['r_squared_adj']
    }

# 循环结束后：跨horizon变量稳定性汇总
print(f"\n{'='*60}")
print("跨 horizon 变量稳定性汇总")
print(f"{'='*60}")

all_features = set()
for h in [6, 12, 36]:
    all_features.update(all_horizon_results[h]['final_features'])

summary_rows = []
for feat in all_features:
    row = {'feature': feat}
    for h in [6, 12, 36]:
        freq_val = all_horizon_results[h]['freq'].get(feat, 0)
        in_final = feat in all_horizon_results[h]['final_features']
        row[f'freq_h{h}'] = round(freq_val, 2)
        row[f'selected_h{h}'] = 'Y' if in_final else '-'
    summary_rows.append(row)

summary_df = pd.DataFrame(summary_rows).sort_values('freq_h12', ascending=False)
print(summary_df.to_string(index=False))



Horizon 6
Best alpha: 0.0002 | Non-zero: 18
=== Bootstrap 稳定性检验 (n_boots=100, threshold=0.9) ===
频率 >= 0.9 的变量: 18 个
频率 >= 0.8 的变量 (高稳定): 29 个
最终进入 OLS 的变量: 10 个

选中频率 top 15:
Element_Name
Explosive Strength             1.00
Dynamic Flexibility            1.00
Fluency of Ideas               0.99
Gross Body Equilibrium         0.99
Memorization                   0.99
Category Flexibility           0.98
Multilimb Coordination         0.97
Number Facility                0.97
Rate Control                   0.97
Oral Comprehension             0.96
Visual Color Discrimination    0.96
Far Vision                     0.95
Wrist-Finger Speed             0.94
Near Vision                    0.93
Dynamic Strength               0.93

R^2: 0.3100 | Adj R^2: 0.2984
Sample size: 609

OLS coefficients:
        O*NET_Activity OLS_Coefficient  CI_Lower  CI_Upper Std_Error P_Value  Sig_10%  Sig_5%
          Memorization         -0.0038 -0.005465 -0.002140  (0.0008)  0.0000     True    True
Multilimb Coord